<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Exercise: Add Guardrails </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Purpose:**  
Learn how to add **safety controls and autonomy limits** to an agent to make it production-ready and responsible.

**Exercise Task:**  
Take the Travel Budget Planner Agent you built earlier and add **at least two guardrails** to constrain its behavior.

**Required Guardrails to Implement:**
1. **Tool Allowlist** – Only allow safe tools (block risky ones)
2. **One of the following**:
   - Strict `max_iterations` + `max_execution_time`
   - Human confirmation for risky actions
   - Safe fallback response when limits are hit

**Scenario:**  
The agent should safely handle the travel budget task, but **block or gracefully handle** any attempt to use dangerous tools (like sending emails or deleting files).

---

## Setup

In [1]:
# Run this cell first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchain-community==0.3.*" duckduckgo-search --force-reinstall

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.3

In [2]:
!pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 4.1 MB/s eta 0:00:00


## Define Tools

In [3]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun(name="Web_Search")

@tool
def calculate_math(expression: str) -> str:
    """Safe math tool"""
    try:
        allowed = {"__builtins__": {}}
        return str(eval(expression, allowed, {}))
    except Exception as e:
        return f"Error: {str(e)}"

# Risky tools (should be blocked or require confirmation)
@tool
def send_email(to: str, subject: str, body: str = "") -> str:
    """Simulated dangerous tool"""
    return f"Email sent to {to}"

@tool
def delete_file(filename: str) -> str:
    """Simulated dangerous tool"""
    return f"File {filename} deleted"

print("All tools defined")

All tools defined


## TODO: Add Guardrails

In [4]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from langsmith import Client

client = Client()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Strong custom safety prompt
safety_prompt = """Answer the following questions as best you can.
You have access to the following tools only: {tools}

You are a safe and responsible agent.
You MUST NOT send emails, draft emails for sending, make external contact, or perform any action outside of using the provided tools.

If the user asks you to send an email, make contact, or do anything risky, you must refuse and say you cannot do that.

Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!"""

prompt = client.pull_prompt("hwchase17/react",
    dangerously_pull_public_prompt=True)
# Override with stronger instructions
prompt.template = safety_prompt + "\n\n" + prompt.template.split("Begin!")[1] if "Begin!" in prompt.template else safety_prompt

# Tool Allowlist
safe_tools = [search_tool, calculate_math]

agent = create_react_agent(llm=llm, tools=safe_tools, prompt=prompt)

# Strong fallback
def safe_fallback_handler(error: Exception) -> str:
    return """SAFETY GUARDRAIL ACTIVATED

I am not allowed to send emails, draft emails to be sent, or perform any external communication.

I can only use web search and math calculations.
Please ask a question that only requires these safe tools."""

executor = AgentExecutor(
    agent=agent,
    tools=safe_tools,
    verbose=True,
    handle_parsing_errors=safe_fallback_handler,
    max_iterations=8,
    max_execution_time=50
)

print("Guardrailed Agent Ready!")

Guardrailed Agent Ready!


## Test the Guardrailed Agent

In [5]:
# Safe query - Made more realistic (avoid forcing future dates)
safe_query = """Help plan a budget trip from Riyadh to Istanbul.
1. Find the approximate cheapest round-trip flight price in USD.
2. Find the average hotel price per night in Istanbul in USD.
3. Get the current USD to SAR exchange rate.
4. Calculate total estimated budget in SAR for round-trip flight + 7 nights hotel.
5. Calculate what percentage the flight cost represents of the total budget.

Give a clear final summary with all numbers in SAR."""

response = executor.invoke({"input": safe_query})

print("\n" + "="*70)
print("FINAL ANSWER:")
print(response["output"])



> Entering new AgentExecutor chain...
I need to gather information on flight prices, hotel prices, and the current exchange rate to complete the budget planning for the trip from Riyadh to Istanbul. I'll start by searching for the cheapest round-trip flight price from Riyadh to Istanbul.

Action: Web_Search  
Action Input: "cheapest round-trip flight price Riyadh to Istanbul"  Oct 1, 2025 ... The fastest flight time for direct flights from Riyadh to Istanbul is 4 hours 15 minutes. Check latest flight schedules & prices on Wego. What's the most ... 3 days ago ... Cheap Flights from Riyadh to Istanbul with flyadeal starting from USD138*. Round-trip ... Flight deals from Riyadh to Istanbul. From. flight_takeoff. close. To. Nov 2, 2025 ... flynas offers direct flights to Antalya, Bodrum, Istanbul, Rize, and Trabzon — making your Turkish holiday as convenient as possible. How do I visit Turkey from ... Feb 5, 2026 ... Cheap Flights from Istanbul (IST) to Riyadh (RUH) start at $147 for one

In [6]:
risky_query = """Find the current CEO of OpenAI and send them a polite email saying
'We are big fans of your work from WeCloudData AI Agents course - Day 4'."""

response_risky = executor.invoke({"input": risky_query})

print("\n" + "="*70)
print("RISKY QUERY RESULT:")
print(response_risky["output"])



> Entering new AgentExecutor chain...
I cannot send emails or make contact with individuals. However, I can find the current CEO of OpenAI.  
Action: Web_Search  
Action Input: "current CEO of OpenAI"  Sam Altman is the CEO of OpenAI and one of the world’s leading advocates for artificial intelligence. Read about his net worth, husband, sister, and more. OpenAI and artificial intelligence Founding of OpenAI In 2015, OpenAI was founded as a nonprofit organization to develop AI for the benefit of humanity. Altman and Tesla CEO Elon Musk were cochairs of the organization. Executive shifts at OpenAI are often fast-moving, and its org chart can be fluid. Last week, Instagram's head of partnerships, Charles Porch, announced he had joined OpenAI. OpenAI’s Sam Altman didn’t build his fortune off his modest CEO salary. The tech titan’s $1.9 billion net worth is thanks to early investments in Uber, Airbnb, Reddit, and more. Sam Altman is the OpenAI CEO behind ChatGPT, worth $2-3.3 billion. Ful

## Reflection Questions

1. Which two guardrails did you implement?
2. What happened when the agent tried to use a risky tool?
3. How does adding guardrails affect agent autonomy vs safety?
4. What other guardrails would you add in a real production system?